## Using RNNs to classify sentiment on IMDB data

In this assignment,you will train three types of RNNs:  "vanilla" RNN, LSTM and GRU to predict the sentiment on IMDB reviews.  

Keras provides a convenient interface to load the data and immediately encode the words into integers (based on the most common words). 
This will save you a lot of the drudgery that is usually involved when working with raw text.

The IMDB is  data consists of 25000 training sequences and 25000 test sequences. 
The outcome is binary (positive/negative) and both outcomes are equally represented in both the training and the test set.


Walk through the followinng steps to prepare the data and the building of an RNN model. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tensorflow as tf
from tensorflow.keras import Sequential, layers, optimizers, initializers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences


tf.keras.utils.set_random_seed(42)


1- Use the `imdb.load_data()` to load in the data 

2- Specify the maximum length of a sequence to 30 words and the pick the 2000 most common words. 

In [ ]:
max_features = 2000
maxlen = 30

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print('x_train length:', len(x_train))
print('x_test length:', len(x_test))
print('sample train sequence:', x_train[0][:20])
print('label:', y_train[0])


3- Check that the number of sequences in train and test datasets are equal (default split):
    
Expected output:
- `x_train = 25000 train sequences`

- `x_test = 25000 test sequences`

In [ ]:
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print('train labels distribution:', {label: int(np.sum(y_train == label)) for label in np.unique(y_train)})
print('test labels distribution:', {label: int(np.sum(y_test == label)) for label in np.unique(y_test)})


4- Pad (or truncate) the sequences so that they are of the maximum length

In [ ]:
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

print('padded x_train shape:', x_train.shape)
print('padded x_test shape:', x_test.shape)


5- After padding or truncating, check the dimensionality of x_train and x_test.

Expected output:
- `x_train shape: (25000, 30)`
- `x_test shape: (25000, 30)`

In [ ]:
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)


In [ ]:
model = Sequential([
    layers.Embedding(input_dim=max_features, output_dim=50, input_length=maxlen),
    layers.SimpleRNN(5, kernel_initializer=initializers.TruncatedNormal(stddev=0.001), activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## Keras layers for (Vanilla) RNNs

In this step, you will not use pre-trained word vectors, Instead you will learn an embedding as part of the  the Vanilla) RNNs network  Neural Network. 

In the Keras API documentation, the Embedding Layer and the SimpleRNN Layer have the following syntax:

### Embedding Layer
`keras.layers.embeddings.Embedding(input_dim, output_dim, embeddings_initializer='uniform', embeddings_regularizer=None, activity_regularizer=None, embeddings_constraint=None, mask_zero=False, input_length=None)`

- This layer maps each integer into a distinct (dense) word vector of length `output_dim`.
- Can think of this as learning a word vector embedding "on the fly" rather than using an existing mapping (like GloVe)
- The `input_dim` should be the size of the vocabulary.
- The `input_length` specifies the length of the sequences that the network expects.

### SimpleRNN Layer
`keras.layers.recurrent.SimpleRNN(units, activation='tanh', use_bias=True, kernel_initializer='glorot_uniform', recurrent_initializer='orthogonal', bias_initializer='zeros', kernel_regularizer=None, recurrent_regularizer=None, bias_regularizer=None, activity_regularizer=None, kernel_constraint=None, recurrent_constraint=None, bias_constraint=None, dropout=0.0, recurrent_dropout=0.0)`

- This is the basic RNN, where the output is also fed back as the "hidden state" to the next iteration.
- The parameter `units` gives the dimensionality of the output (and therefore the hidden state).  Note that typically there will be another layer after the RNN mapping the (RNN) output to the network output.  So we should think of this value as the desired dimensionality of the hidden state and not necessarily the desired output of the network.
- Recall that there are two sets of weights, one for the "recurrent" phase and the other for the "kernel" phase.  These can be configured separately in terms of their initialization, regularization, etc.






6- Build the RNN with three layers: 
- The SimpleRNN layer with 5 neurons and initialize its kernel with stddev=0.001

- The Embedding layer and initialize it by setting the word embedding dimension to 50. This means that this layer takes each integer in the sequence and embeds it in a 50-dimensional vector.

-  The output layer has the sigmoid activation function.

In [ ]:
model = Sequential([
    layers.Embedding(input_dim=max_features, output_dim=50, input_length=maxlen),
    layers.SimpleRNN(5, kernel_initializer=initializers.TruncatedNormal(stddev=0.001), activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


7- How many parameters have the embedding layer?

In [ ]:
embedding_layer = model.layers[0]
num_params = embedding_layer.input_dim * embedding_layer.output_dim
print('Number of parameters in the embedding layer:', num_params)


8- Train the network with the RMSprop with learning rate of .0001 and epochs=10.

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    verbose=1
)


In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.title('Accuracy over epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', loss)
print('Test accuracy:', accuracy)


9- PLot the loss and accuracy metrics during the training and interpret the result.

In [ ]:
# Optional: inspect a few predictions
preds = (model.predict(x_test[:10], verbose=0).ravel() > 0.5).astype(int)
for i, (p, y) in enumerate(zip(preds, y_test[:10])):
    print(i, 'prediction=', p, 'label=', y)


10- Check the accuracy and the loss of your models on the test dataset.

In [ ]:
# Evaluate the trained model on the full test set
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', loss)
print('Test accuracy:', accuracy)


## Tuning The Vanilla RNN Network


11- Prepare the data to use sequences of length 80 rather than length 30 and retrain your model.  Did it improve the performance?

12- Try different values of the  maximum length of a sequence ("max_features").  Can you improve the performance?

13- Try smaller and larger sizes of the RNN hidden dimension.  How does it affect the model performance?  How does it affect the run time?

In [ ]:
# Rebuild with longer sequences for the tuning experiment
maxlen_80 = 80
x_train_80 = pad_sequences(x_train, maxlen=maxlen_80)
x_test_80 = pad_sequences(x_test, maxlen=maxlen_80)

model_80 = Sequential([
    layers.Embedding(input_dim=max_features, output_dim=50, input_length=maxlen_80),
    layers.SimpleRNN(5, kernel_initializer=initializers.TruncatedNormal(stddev=0.001), activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])
model_80.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

history_80 = model_80.fit(x_train_80, y_train, validation_split=0.2, epochs=5, batch_size=128, verbose=0)
loss_80, acc_80 = model_80.evaluate(x_test_80, y_test, verbose=0)
print('maxlen=80 test accuracy:', acc_80)


## Train LSTM and GRU networks


14- Build LSTM and GRU networks and compare their performance (accuracy and execution time) with the SimpleRNN. What is your conclusion?

In [ ]:
def build_and_train(model_name, layer_type, units, maxlen=maxlen, epochs=3):
    model = Sequential([
        layers.Embedding(input_dim=max_features, output_dim=50, input_length=maxlen),
        layer_type(units, activation='tanh'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizers.RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

    start = time.time()
    history = model.fit(x_train, y_train, validation_split=0.2, epochs=epochs, batch_size=128, verbose=0)
    elapsed = time.time() - start
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(model_name, 'test accuracy:', round(acc, 4), 'time (s):', round(elapsed, 2))
    return acc, elapsed

# Compare the three recurrent layers.
try:
    simple_rnn_acc, simple_rnn_time = build_and_train('SimpleRNN', layers.SimpleRNN, 5, epochs=3)
except Exception as e:
    print('SimpleRNN failed:', e)

try:
    lstm_acc, lstm_time = build_and_train('LSTM', layers.LSTM, 5, epochs=3)
except Exception as e:
    print('LSTM failed:', e)

try:
    gru_acc, gru_time = build_and_train('GRU', layers.GRU, 5, epochs=3)
except Exception as e:
    print('GRU failed:', e)
